# Optimizer bake-off on the kneed walker (Colab)

Compares five search methods at an equal **episode** budget on the 520-weight kneed brain:
`ga`, `cmaes`, `openai_es`, `es_raw` (the literal `theta += alpha/(n*sigma) * sum(F_i * eps_i)` update), and `random_search`.

**Runtime: CPU, not GPU.** The cost is Pymunk physics on the CPU; the 324/520-weight matmul is noise. A GPU runtime works but wastes the GPU.

Order of business: pilot (1/5 seeds, 1/5 budget, ~10 min) -> read the curves -> full run (~4-6 h, resumable if Colab disconnects). The pilot is a plumbing check, not evidence.

## 1. Get the code

If the repo is private, either sign in first (`from google.colab import auth; auth.authenticate_user()`) or upload a zip and unzip it to `/content/Evolution-1`, then skip the clone.

In [1]:
REPO = "https://github.com/000wahab000/Evolution.git"
DIR = "/content/Evolution-1"

import os
if os.path.isdir(os.path.join(DIR, ".git")):
    !git -C {DIR} pull --ff-only
else:
    !git clone {REPO} {DIR}
%cd {DIR}
!git log --oneline -1

d:\content\Evolution-1


Cloning into '/content/Evolution-1'...


2fa38d0 Merge pull request #15 from 000wahab000/claude/evolution-high-detail-summary-418ee3


## 2. Install (research extras: cma, scipy, matplotlib)

In [3]:
!pip -q install -e ".[research]"
!python -c "import numpy, pymunk, cma, scipy; print('numpy', numpy.__version__, '| pymunk', pymunk.version, '| cma', cma.__version__, '| scipy', scipy.__version__)"

  C:\Users\Wahab\AppData\Local\Temp\pip-build-env-29txu1po\overlay\Lib\site-packages\setuptools\command\editable_wheel.py:354: BuildBackendWarning: Editable installation.
  !!
  
          ********************************************************************************
          Please be careful with folders in your working directory with the same
          name as your package as they may take precedence during imports.
          ********************************************************************************
  
  !!
    with strategy, WheelFile(wheel_path, "w") as wheel_obj:


numpy 2.5.4 | pymunk 7.3.1 | cma 4.5.0 | scipy 1.18.1


## 3. Settings

Mount Drive so results survive a disconnect. Each run writes `results/<exp>/<variant>/seed_<n>/`, skips itself once `DONE` exists, and resumes on rerun.

In [5]:
MOUNT_DRIVE = True

import os

WORKERS = len(os.sched_getaffinity(0))  # real vCPUs; Colab's os.cpu_count() is often a lie
print('workers ->', WORKERS)

workers -> 2


## 4. Pilot run (5 seeds x 5,000 episodes per variant)

~25 runs. Expect roughly 10 minutes on 8 vCPUs.

In [6]:
!python scripts/run_experiment.py experiments/pilot5_optimizers_knees.json --workers {WORKERS} --out {OUT}

python3: can't open file '/content/scripts/run_experiment.py': [Errno 2] No such file or directory


## 5. Analyze the pilot, and check the cost before the full run

In [ ]:
!python scripts/analyze.py {OUT}/pilot5_optimizers_knees --no-replay

In [ ]:
import glob, json

def core_hours(exp):
    files = glob.glob(f'{OUT}/{exp}/*/seed_*/meta.json')
    return sum(json.load(open(f))['wall_s'] for f in files) / 3600

pilot_h = core_hours('pilot5_optimizers_knees')
print(f'pilot: {pilot_h:.2f} core-hours')
print(f'full exp5 = 6x the seeds -> {pilot_h*6:.1f} core-hours -> ~{pilot_h*6/WORKERS:.1f} h on {WORKERS} workers')

### Gate: read this before spending the full budget

Open `pilot5_optimizers_knees/summary/REPORT.md` and `summary/figures/F1.png` (learning curves).

- Every variant must show a rising `best_train` curve. A flat line means the plumbing or that optimizer's step size is wrong, not that the method lost.
- `es_raw` must not diverge (curve collapsing to a large negative return). If it does, note it and lower `alpha` in both configs *before* the full run, so the comparison is not a straw man.
- Treat pilot numbers as a cost estimate only. 5 seeds cannot support H11.

## 6. Full run (30 seeds x 25,000 episodes per variant)

150 runs. Re-run this cell after a disconnect; finished runs are skipped. Do not change the configs between the pilot and this run.

In [ ]:
!python scripts/run_experiment.py experiments/exp5_optimizers_knees.json --workers {WORKERS} --out {OUT}

## 7. Analyze and read the verdict

F7/F9 re-simulate the saved champions, so this takes a few extra minutes. `H11` is the pre-registered contrast (openai_es vs es_raw); every other pair is exploratory.

In [ ]:
!python scripts/verify_results.py {OUT}/exp5_optimizers_knees
!python scripts/analyze.py {OUT}/exp5_optimizers_knees

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

display(Markdown('## Summary (IQM + 95% bootstrap CI)'))
s = pd.read_csv(f'{OUT}/exp5_optimizers_knees/summary/summary.csv')
cols = [c for c in ['variant','n','test_distance_iqm','test_distance_ci_lo','test_distance_ci_hi','fall_rate','exploit_flag'] if c in s]
display(s[cols].round(2))

display(Markdown('## Pairwise contrasts'))
p = pd.read_csv(f'{OUT}/exp5_optimizers_knees/summary/pairwise.csv')
cols = [c for c in ['id','a','b','metric','verdict','p_holm','cliffs_delta'] if c in p]
display(p[cols])

In [ ]:
display(Markdown(open(f'{OUT}/exp5_optimizers_knees/summary/REPORT.md').read()))

## 8. Take the results with you

Ship the folder back to the repo (or attach it to the PR). Replays recorded on Colab/Linux can differ from Windows: chaotic physics, same brain.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/exp5_optimizers_knees', 'zip', f'{OUT}/exp5_optimizers_knees')
files.download('/content/exp5_optimizers_knees.zip')